In [1]:
import pandas as pd

In [2]:
data_path = "../data/combined/"
data_file = "amplitude_csi_dataframe.pkl"

DISCRETE_VARIABLES = ["person"]
TARGET_VARIABLE = "position"
STATE = 42

min_subcarrier = 0
max_subcarrier = 60

data_df: pd.DataFrame = pd.read_pickle(data_path + data_file)
columns_to_drop = [
    col
    for col in data_df.columns
    if isinstance(col, (int)) and (col > max_subcarrier or col < min_subcarrier)
]

data_df.drop(columns=columns_to_drop, inplace=True)
total_columns = len(data_df.columns)

# Convert all column names to strings
data_df.columns = data_df.columns.astype(str)

print(total_columns)
print(data_df.columns)
print(data_df.head())

55
Index(['person', 'position', '6', '7', '8', '9', '10', '11', '12', '13', '14',
       '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '26', '27',
       '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39',
       '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51',
       '52', '54', '55', '56', '57', '58', '59', '60'],
      dtype='object')
   person  position            6            7            8            9  \
0       1        17   795.910156   849.388000   890.166809   912.882263   
1       1        17   798.279419   843.614258   868.484314   895.013977   
2       1        17  1064.543091  1086.945312  1105.320312  1135.975342   
3       1        17  1060.771362  1092.156128  1112.137573  1130.086670   
4       1        17  1329.939087  1409.457397  1416.469604  1432.482056   

            10           11           12           13  ...           50  \
0   946.926086   979.547363  1059.871704  1146.253052  ...  1196.047607   
1 

In [3]:
impact_matrix_file = "../data/impact_matrix/sorted_reduced_impact_matrix.csv"

POSITIONS = 18

subcarrier_limit = 40

sorted_subcarrier_impact = pd.read_csv(impact_matrix_file, index_col=0)

position_data: dict[int, pd.DataFrame] = {
    i: data_df.drop(
        list(map(str, sorted_subcarrier_impact.iloc[subcarrier_limit:, i].to_list())),
        axis=1,
    )
    for i in range(POSITIONS)
}

print(len(position_data[0].columns))
print(position_data[0].columns)
print(position_data[0].head())

42
Index(['person', 'position', '6', '7', '8', '9', '10', '11', '14', '15', '16',
       '17', '18', '19', '23', '24', '26', '27', '28', '29', '31', '34', '36',
       '37', '40', '41', '42', '43', '44', '45', '46', '47', '50', '51', '52',
       '54', '55', '56', '57', '58', '59', '60'],
      dtype='object')
   person  position            6            7            8            9  \
0       1        17   795.910156   849.388000   890.166809   912.882263   
1       1        17   798.279419   843.614258   868.484314   895.013977   
2       1        17  1064.543091  1086.945312  1105.320312  1135.975342   
3       1        17  1060.771362  1092.156128  1112.137573  1130.086670   
4       1        17  1329.939087  1409.457397  1416.469604  1432.482056   

            10           11           14           15  ...           50  \
0   946.926086   979.547363  1273.367554  1415.204590  ...  1196.047607   
1   921.195984   935.745667  1217.236572  1353.547241  ...  1047.532837   
2  1170.0299

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

X_train, X_test, y_train, y_test = {}, {}, {}, {}

for i in range(POSITIONS):
    numerical_columns = [
        col
        for col in position_data[i].columns
        if col not in DISCRETE_VARIABLES and col != TARGET_VARIABLE
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", "passthrough", DISCRETE_VARIABLES),
            ("num", numeric_transformer, numerical_columns),
        ]
    )

    X = position_data[i].drop(columns=TARGET_VARIABLE)
    X = preprocessor.fit_transform(X)
    y = position_data[i][TARGET_VARIABLE] == i

    X_train[i], X_test[i], y_train[i], y_test[i] = train_test_split(
        X, y, test_size=0.2, random_state=STATE
    )

print(X_train[0][:5])
print(y_train[0][:5])
print(X_test[0][:5])
print(y_test[0][:5])

[[ 4.4000000e+01  3.8642046e-01  2.9976055e-01  2.1416575e-01
   1.3459538e-01  4.1206248e-02 -5.1461264e-02 -1.7905062e-01
  -2.1703133e-01 -2.0290710e-01 -1.9349921e-01 -1.5718175e-01
  -3.3163317e-02  5.2003330e-01  7.1807045e-01  1.1197482e+00
   1.3428845e+00  1.5217314e+00  1.7500858e+00  1.8880272e+00
   2.1979859e+00  1.9847615e+00  1.8393641e+00  1.4779557e+00
   1.4554825e+00  1.1440097e+00  9.0687943e-01  7.6222944e-01
   4.3166617e-01  2.0760611e-01  1.8334143e-02 -4.3697897e-01
  -5.8092111e-01 -7.2828388e-01 -1.0800629e+00 -1.2078531e+00
  -1.2877849e+00 -1.4551461e+00 -1.5468431e+00 -5.2356905e-01
  -5.6683284e-01]
 [ 2.6000000e+01 -6.9760364e-01 -6.8234670e-01 -6.4446419e-01
  -6.4048821e-01 -6.1059684e-01 -5.8075351e-01 -5.2101505e-01
  -4.8338830e-01 -4.4629633e-01 -4.3466631e-01 -4.0380964e-01
  -3.5520279e-01 -2.0835596e-01 -1.3066614e-01  5.8470488e-02
   2.0658769e-01  2.9838407e-01  4.7207931e-01  6.6079527e-01
   8.7452042e-01  1.0225917e+00  1.0569628e+00  1.03

In [5]:
save_path = "../data/filtered_train_test_split/"


def save_pkl(obj: object, path: str) -> None:
    with open(path, "wb") as f:
        pd.to_pickle(obj, f)


for i in range(POSITIONS):
    save_pkl(X_train[i], save_path + f"X_train_{i}.pkl")
    save_pkl(y_train[i], save_path + f"y_train_{i}.pkl")
    save_pkl(X_test[i], save_path + f"X_test_{i}.pkl")
    save_pkl(y_test[i], save_path + f"y_test_{i}.pkl")

print("Data saved to", save_path)

Data saved to ../data/filtered_train_test_split/


In [6]:
import torch

for i in range(POSITIONS):
    X_train_tensor = torch.tensor(X_train[i], dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train[i].values, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test[i], dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test[i].values, dtype=torch.long)

    torch.save(X_train_tensor, save_path + f"X_train_{i}.pt")
    torch.save(y_train_tensor, save_path + f"y_train_{i}.pt")
    torch.save(X_test_tensor, save_path + f"X_test_{i}.pt")
    torch.save(y_test_tensor, save_path + f"y_test_{i}.pt")

print("Data saved to", save_path)

Data saved to ../data/filtered_train_test_split/


In [7]:
train_X = torch.load(save_path + "X_train_0.pt")
train_y = torch.load(save_path + "y_train_0.pt")
test_X = torch.load(save_path + "X_test_0.pt")
test_y = torch.load(save_path + "y_test_0.pt")

print(train_X[:5])
print(train_y[:5])
print(test_X[:5])
print(test_y[:5])

tensor([[ 4.4000e+01,  3.8642e-01,  2.9976e-01,  2.1417e-01,  1.3460e-01,
          4.1206e-02, -5.1461e-02, -1.7905e-01, -2.1703e-01, -2.0291e-01,
         -1.9350e-01, -1.5718e-01, -3.3163e-02,  5.2003e-01,  7.1807e-01,
          1.1197e+00,  1.3429e+00,  1.5217e+00,  1.7501e+00,  1.8880e+00,
          2.1980e+00,  1.9848e+00,  1.8394e+00,  1.4780e+00,  1.4555e+00,
          1.1440e+00,  9.0688e-01,  7.6223e-01,  4.3167e-01,  2.0761e-01,
          1.8334e-02, -4.3698e-01, -5.8092e-01, -7.2828e-01, -1.0801e+00,
         -1.2079e+00, -1.2878e+00, -1.4551e+00, -1.5468e+00, -5.2357e-01,
         -5.6683e-01],
        [ 2.6000e+01, -6.9760e-01, -6.8235e-01, -6.4446e-01, -6.4049e-01,
         -6.1060e-01, -5.8075e-01, -5.2102e-01, -4.8339e-01, -4.4630e-01,
         -4.3467e-01, -4.0381e-01, -3.5520e-01, -2.0836e-01, -1.3067e-01,
          5.8470e-02,  2.0659e-01,  2.9838e-01,  4.7208e-01,  6.6080e-01,
          8.7452e-01,  1.0226e+00,  1.0570e+00,  1.0350e+00,  1.0508e+00,
          1.013

C:\Users\gurgel\AppData\Local\Temp\ipykernel_16892\3029730503.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_X = torch.load(save_path + "X_train_0.pt")
C:\Users\g

tensor([0, 0, 0, 0, 0])
tensor([[ 4.6000e+01, -2.1376e-01, -2.7302e-01, -2.0079e-01, -2.5825e-01,
         -2.2256e-01, -2.2331e-01, -2.6441e-01, -2.9630e-01, -3.0502e-01,
         -3.1973e-01, -3.3386e-01, -4.1456e-01, -7.1675e-01, -8.0627e-01,
         -9.0755e-01, -1.0729e+00, -1.0910e+00, -1.0705e+00, -1.1417e+00,
         -1.1264e+00, -1.0887e+00, -1.0940e+00, -1.0091e+00, -9.9629e-01,
         -9.2584e-01, -8.4365e-01, -9.1054e-01, -8.1281e-01, -7.9621e-01,
         -6.8563e-01, -6.5276e-01, -6.0489e-01, -6.4545e-01, -5.5362e-01,
         -5.5409e-01, -5.8791e-01, -5.9253e-01, -5.4771e-01, -1.1744e+00,
         -1.1432e+00],
        [ 1.0000e+01, -1.2046e+00, -1.1377e+00, -1.1721e+00, -1.0619e+00,
         -9.7694e-01, -9.8520e-01, -8.2514e-01, -7.6225e-01, -7.4898e-01,
         -6.0825e-01, -7.8904e-01, -7.1147e-01, -6.8148e-01, -5.9219e-01,
         -6.5168e-01, -5.0646e-01, -4.5720e-01, -3.8838e-01, -3.0352e-01,
          6.8895e-02,  2.3324e-01,  4.0634e-01,  5.7931e-01,  6.1